In [ ]:
import torch
import torch.nn as nn

In [ ]:
torch.manual_seed(123)
batch_example = torch.randn(2, 5) #A
layer = nn.Sequential(nn.Linear(5, 6), nn.ReLU())
out = layer(batch_example)
print(out)

In [ ]:
class multi_head_attention(torch.nn.Module):
    def __init__(self, d_in,d_out,context_length,drop_out,num_heads,qkv_bias=False):
        super().__init__()
        
        assert (d_out%num_heads==0), \
            "d_out must be divisible by number of heads"
            
        self.d_out=d_out
        self.d_in=d_in
        self.head_dim=d_out//num_heads
        self.num_heads=num_heads
        self.out_proj=torch.nn.Linear(d_out, d_out)
        
        self.W_query=torch.nn.Linear(d_in,d_out, bias=qkv_bias)
        self.W_key=torch.nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_value=torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        
        self.dropout=torch.nn.Dropout(drop_out)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length),diagonal=1) )
        
    def forward(self,x):
        b,num_tokens,d_in=x.shape
        keys=self.W_key(x)
        values=self.W_value(x)
        query=self.W_query(x)
        
        # Now we split the b,num_tokens,d_out to b, num_tokens, num_heads, head_dim
        
        keys=keys.view(b,num_tokens,self.num_heads,self.head_dim)
        values=values.view(b, num_tokens, self.num_heads, self.head_dim)
        query=query.view(b, num_tokens, self.num_heads, self.head_dim)
        
        # Now group the matrices by head
        
        keys=keys.transpose(1,2) # b, num_heads, num_tokens, head_dim
        values=values.transpose(1,2)
        query=query.transpose(1,2)
        
        attention_scores = query @ keys.transpose(2,3)
        mask_bool=self.mask.bool()[:num_tokens,:num_tokens]
        attention_scores.masked_fill_(mask_bool, float('-inf'))
        
        attention_weights = torch.softmax(attention_scores/keys.shape[-1]**0.5, dim=-1)
        attention_weights=self.dropout(attention_weights) # shape is b, num_head,num_tokens,num_tokens
        context_vector =(attention_weights @ values).transpose(1,2)
        context_vector=context_vector.contiguous().view(b,num_tokens,self.d_out)
        context_vector= self.out_proj(context_vector)
        return context_vector
        

In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 256, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

In [ ]:
class DummyGptModel(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.token_emb=nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb=nn.Embedding(cfg["context_length"],cfg["emb_dim"])
        self.drop_emb=nn.Dropout(cfg["drop_rate"])
        
        self.trf_block=nn.Sequential(
            *[transformer_block(cfg) for _ in range(cfg["n_layers"])]
        )
        
        self.final_norm= LayerNorm(cfg["emb_dim"])
        self.out_head=nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )
        
    def forward(self,in_idx):
        batch_size, seq_len=in_idx.shape
        token_emb=self.token_emb(in_idx)
        pos_emb=self.pos_emb(torch.arange(seq_len,device=in_idx.device))
        x=token_emb + pos_emb
        x=self.drop_emb(x)
        x=self.trf_block(x)
        x=self.final_norm(x)
        logits=self.out_head(x)
        return logits
class transformer_block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # A simple placeholder

    def forward(self, x):
        # This block does nothing and just returns its input.
        return x


class LayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()
        self.eps=eps
        self.scale=nn.Parameter(torch.ones(normalized_shape))
        self.shift=nn.Parameter(torch.zeros(normalized_shape))

    def forward(self, x):
        # This layer does nothing and just returns its input.
        mean=torch.mean(x,dim=-1,keepdim=True)
        var=torch.var(x,dim=-1,unbiased=False,keepdim=True)
        x=(x-mean)/torch.sqrt(var+self.eps)
        x=self.scale*x+self.shift
        return x

        
            
        
        

In [ ]:
ln = LayerNorm(normalized_shape=5)
out_ln = ln(batch_example)
mean = out_ln.mean(dim=-1, keepdim=True)
var = out_ln.var(dim=-1, unbiased=False, keepdim=True)
print("Mean:\n", mean)
print("Variance:\n", var)

In [ ]:
class GeluActivation(nn.Module):
    def __init__(self):
        super().__init__()
        
    def forward(self,x):
        return 0.5*x*(1+torch.tanh(torch.sqrt(torch.tensor(2/torch.pi))*(x+0.044715*torch.pow(x,3))))

In [ ]:
import matplotlib.pyplot as plt
relu=nn.ReLU()
gelu=GeluActivation()
x=torch.linspace(-3,3,100)
y_relu,y_gelu=relu(x), gelu(x)

plt.figure(figsize=(8,3))
for i,(y,label) in enumerate(zip([y_gelu,y_relu],["GELU","RELU"]),1):
    plt.subplot(1,2,i)
    plt.plot(x,y)
    plt.title(f"{label} activation function")
    plt.xlabel("Input value (x)")
    plt.ylabel(f"{label} output")
    plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
class feedforward(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.layers=nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4*cfg["emb_dim"]),
            GeluActivation(),
            nn.Linear(4*cfg["emb_dim"],cfg["emb_dim"]),
            
        )
        
    def forward(self,x):
      return self.layers(x)

In [ ]:
print(GPT_CONFIG_124M["emb_dim"])

In [ ]:
ffn=feedforward(cfg=GPT_CONFIG_124M)
x = torch.rand(2, 3, 768)
ffn_out=ffn(x)
print("Input shape:", x.shape)
print("Output shape:", ffn_out.shape)

In [ ]:
class shorcut_connetction(nn.Module):
    def __init__(self,layers_size,shortcut):
        super().__init__()
        self.shortcut=shortcut
        self.layers=nn.Sequential(
            nn.Linear(layers_size[0],layers_size[1]),
            GeluActivation(),
            nn.Linear(layers_size[1],layers_size[2]),
            GeluActivation(),
            nn.Linear(layers_size[2],layers_size[3]),
            GeluActivation(),
            nn.Linear(layers_size[3],layers_size[4]),
            GeluActivation(),
            nn.Linear(layers_size[4], layers_size[5])
        )
    
    def forward(self,x):
        layer_out=self.layers(x)
        if self.shortcut:
            x=x+layer_out
        else:
            x=layer_out
        return x

        

In [ ]:
layer_sizes = [3, 3, 3, 3, 3, 1]
sample_input = torch.tensor([[1., 0., -1.]])
torch.manual_seed(123) # specify random seed for the initial weights for reproducibility
model_without_shortcut = shorcut_connetction(
layer_sizes, shortcut=False
)
model_with_shortcut_connection=shorcut_connetction(layer_sizes,shortcut=True)

In [ ]:
def print_gradient(model,x):
    out=model(x)
    target = torch.tensor([[0.]])
    loss_fn=nn.MSELoss()
    loss=loss_fn(out,target)
    loss.backward()
    
    for name,parameter in model.named_parameters():
        if "weight" in name:
            print(f"Gradient of {name}: {parameter.grad.mean().item()}")

In [ ]:
print_gradient(model_without_shortcut, sample_input)

In [ ]:
print_gradient(model_with_shortcut_connection,sample_input)

GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}
 d_in,d_out,context_length,drop_out,num_heads,qkv_bias=False

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.attention=multi_head_attention(
                                       
                                             d_in=cfg["emb_dim"],
                                             d_out=cfg["emb_dim"],
                                             context_length=cfg["context_length"],
                                             num_heads=cfg["n_heads"],
                                             drop_out=cfg["drop_rate"],
                                             qkv_bias=cfg["qkv_bias"])
        self.layer_norm1=nn.LayerNorm(normalized_shape=cfg["emb_dim"])
        self.layer_norm2=nn.LayerNorm(normalized_shape=cfg["emb_dim"])
        self.ffn=feedforward(cfg)
        self.drop_shorcut=nn.Dropout(cfg["drop_rate"])
        
    def forward(self,x):
        shortcut=x
        x=self.layer_norm1(x)
        x=self.attention(x)
        x=self.drop_shorcut(x)
    
        x=x+shortcut
        shortcut=x
        x=self.layer_norm2(x)
        x=self.ffn(x)
        x=self.drop_shorcut(x)
        x=x+shortcut
        
        return x
                
        
        

In [ ]:
torch.manual_seed(123)
x=torch.randn(2,3,768)
cfg=GPT_CONFIG_124M
transformer_block=TransformerBlock(cfg)
out=transformer_block(x)
print("Input shape:", x.shape)
print("Output shape:", out.shape)

In [ ]:
class GPTModule(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.token_emb=nn.Embedding(cfg["vocab_size"],cfg["emb_dim"])
        self.pos_dim=nn.Embedding(cfg["context_length"],cfg["emb_dim"])
        self.drop=nn.Dropout(cfg["drop_rate"])
        self.trf_blocks=nn.Sequential(
            *[TransformerBlock(cfg) for _ in range (cfg["n_layers"])]
        )
        self.final_norm=nn.LayerNorm(cfg["emb_dim"])
        self.out_head=nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)
        
    def forward(self,in_idx):
        batch_size,seq_len=in_idx.shape
        token_emb=self.token_emb(in_idx)
        pos_emb=self.pos_dim(torch.arange(seq_len, device=in_idx.device))
        x=token_emb+pos_emb
        
        x=self.drop(x)
        x=self.trf_blocks(x)
        x=self.final_norm(x)
        logits=self.out_head(x)
        return logits

        
    

In [ ]:
batch=torch.tensor([
    [0.43,0.15,0.8,0.55,0.87,0.66], # your
    [0.57,0.85,0.64,0.22,0.58,0.23], # journey
    [0.77,0.25,0.10,0.05,0.80,0.55], # starts
])

In [ ]:
torch.manual_seed(123)
model = GPTModule(GPT_CONFIG_124M)
out = model(batch.long())
print("Input batch:\n", batch)
print("\nOutput shape:", out.shape)
print(out)

In [ ]:
total_parameter=sum(p.numel() for p in model.parameters())
print(f"Total parameter {total_parameter:,}")

In [ ]:
print("Token embedding layer shape:", model.token_emb.weight.shape)
print("Output layer shape:", model.out_head.weight.shape)

In [ ]:
total_size_bytes = total_parameter * 4 #A
total_size_mb = total_size_bytes / (1024 * 1024) #B
print(f"Total size of the model: {total_size_mb:.2f} MB")

In [ ]:
def generate_simpel_text(model, idx, context_size,max_new_token):
    for _ in range (max_new_token):
        idx_cond=idx[:,-context_size:]
        with torch.no_grad():
            logits=model(idx_cond)
        logits=logits[:,-1,:]
        probas=torch.softmax(logits,dim=-1)
        next_idx=torch.argmax(probas, dim=-1, keepdim=True)
        idx=torch.cat((idx, next_idx), dim=1)
    return idx


In [ ]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))
tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
start_context = "My name"
encoded = tokenizer.encode(start_context)
print("encoded:", encoded)
encoded_tensor = torch.tensor(encoded).unsqueeze(0) #A
print("encoded_tensor.shape:", encoded_tensor.shape)

In [ ]:
model.eval() #A
out = generate_simpel_text(
model=model,
idx=encoded_tensor,
max_new_token=10,
context_size=GPT_CONFIG_124M["context_length"]
)
print("Output:", out)
print("Output length:", len(out[0]))

In [ ]:
decoded_text = tokenizer.decode(out.squeeze(0).tolist())
print(decoded_text)

In [ ]:
inputs = torch.tensor([[16833, 3626, 6100],   # ["every effort moves",
                       [40,    1107, 588]])   #  "I really like"]

targets = torch.tensor([[3626, 6100, 345  ],  # [" effort moves you",
                        [1107,  588, 11311]]) #  " really like chocolate"]

In [ ]:
with torch.no_grad():
    logits=model(inputs)
    
prob=torch.softmax(logits,dim=-1)
print(prob.shape)


In [ ]:
token_id=torch.argmax(prob,dim=-1,keepdim=True)
print("Token IDs:\n", token_id)

In [ ]:
import tiktoken
import tiktoken

def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # add batch dimension
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # remove batch dimension
    return tokenizer.decode(flat.tolist())

start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")




In [ ]:
print(f"Targets batch 1: {token_ids_to_text(targets[0], tokenizer)}")
print(f"Outputs batch 1: {token_ids_to_text(token_id[0].flatten(), tokenizer)}")

In [ ]:
text_idx=0
target_prob_1=prob[text_idx,[0,1,2],targets[text_idx]]

text_idx=1
target_prob_2=prob[text_idx,[0,1,2],targets[text_idx]]

print(f"Target prbabilites 1 is {target_prob_1} \n Target probabiblty 2 is {target_prob_2}")


In [ ]:
log_prob=torch.log(torch.cat((target_prob_1,target_prob_2)))
mean_log_prob=log_prob.mean()
mean_log_prob=-mean_log_prob
mean_log_prob

In [ ]:
print(f"logits shape is {logits.shape}")
print(f"targets sahpe is {targets.shape}")


In [ ]:
flatten_logits=torch.flatten(logits, 0, 1  )
target_flatten=torch.flatten(targets)
loss=torch.nn.functional.cross_entropy(
flatten_logits, target_flatten)
loss

In [ ]:
perplexity = torch.exp(loss)
print(perplexity)

In [ ]:
file_path="story.text"
with open(file_path, "r") as f:
    text_data=f.read()

In [ ]:
total_characters = len(text_data)
total_tokens = len(tokenizer.encode(text_data))

print("Characters:", total_characters)
print("Tokens:", total_tokens)

In [ ]:
from torch.utils.data import DataLoader, Dataset
class GPTDatasetV1(Dataset):
    def __init__(self,text,tokenizer,max_length,stride):
        self.input_ids=[]
        self.output_ids=[]
        # tokenize the entire text
        token_ids=tokenizer.encode(text)
        
        for i in range(0,len(token_ids)-max_length,stride):
            input_chunk=token_ids[i:i+max_length]
            output_chunk=token_ids[i+1: i+1+max_length]
            self.input_ids.append(torch.tensor(input_chunk))
            self.output_ids.append(torch.tensor(output_chunk))
            
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return self.input_ids[idx],self.output_ids[idx]
    
def create_dataloaderV1(text,batch_size=4,max_len=256,stride=128,shuffle=True,drop_last=True,num_workers=0):
    tokenizer=tiktoken.get_encoding("gpt2")
    dataset=GPTDatasetV1(text,tokenizer,max_len,stride)
    
    data_loader=DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
        
    )
    return data_loader

    

In [ ]:
train_ratio=0.9
train_idx=int(train_ratio*len(text_data))
train_data=text_data[:train_idx]
val_data=text_data[train_idx:]

torch.manual_seed(123)

train_loader=create_dataloaderV1(
    train_data,
    batch_size=2,
    max_len=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    shuffle=True
)
val_loader = create_dataloaderV1(
    val_data,
    batch_size=2,
    max_len=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    
)

if total_tokens * (train_ratio) < GPT_CONFIG_124M["context_length"]:
    print("Not enough tokens for the training loader. "
          "Try to lower the `GPT_CONFIG_124M['context_length']` or "
          "increase the `training_ratio`")

if total_tokens * (1-train_ratio) < GPT_CONFIG_124M["context_length"]:
    print("Not enough tokens for the validation loader. "
          "Try to lower the `GPT_CONFIG_124M['context_length']` or "
          "decrease the `training_ratio`")

In [ ]:
print("Train loader:")
for x, y in train_loader:
    print(x.shape, y.shape)

print("\nValidation loader:")
for x, y in val_loader:
    print(x.shape, y.shape)

In [ ]:
train_tokens = 0
for input_batch, target_batch in train_loader:
    train_tokens += input_batch.numel()

val_tokens = 0
for input_batch, target_batch in val_loader:
    val_tokens += input_batch.numel()

print("Training tokens:", train_tokens)
print("Validation tokens:", val_tokens)
print("All tokens:", train_tokens + val_tokens)

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # Reduce the number of batches to match the total number of batches in the data loader
        # if num_batches exceeds the number of batches in the data loader
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

In [ ]:

if torch.cuda.is_available():
   device = torch.device("cuda")
elif torch.backends.mps.is_available():
   device = torch.device("mps")
else:
   device = torch.device("cpu")

print(f"Using {device} device.")


model.to(device) # no assignment model = model.to(device) necessary for nn.Module classes


torch.manual_seed(123) # For reproducibility due to the shuffling in the data loader

with torch.no_grad(): # Disable gradient tracking for efficiency because we are not training, yet
    train_loss = calc_loss_loader(train_loader, model, device)
    val_loss = calc_loss_loader(val_loader, model, device)

print("Training loss:", train_loss)
print("Validation loss:", val_loss)

In [ ]:
inputs = torch.tensor([[16833, 3626, 6100],   # ["every effort moves",
                       [40,    1107, 588]])   #  "I really like"]

targets = torch.tensor([[3626, 6100, 345  ],  # [" effort moves you",
                        [1107,  588, 11311]]) #  " really like chocolate"]

In [ ]:
def evaluate_model(model,train_loader,val_loader,device,eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss=calc_loss_loader(train_loader,model,device,eval_iter)
        val_loss=calc_loss_loader(val_loader,model, device,eval_iter)
    model.train()
    return train_loss, val_loss
    
    

In [ ]:
def generate_and_print_sample(model,tokenize,device, start_context):
    model.eval()
    context_size=model.pos_dim.weight.shape[0]
    encoded=text_to_token_ids(start_context,tokenize).to(device)
    with torch.no_grad():
        tokens_ids=generate_simpel_text(model,encoded,context_size,50)
    decoded_text = token_ids_to_text(tokens_ids, tokenizer)
    print(decoded_text.replace("\n", " "))  # Compact print format
    model.train()
    

In [ ]:
def train_simple_model(model,
                       train_loader,
                       val_loader,
                       optimizer,
                       device,
                       epochs,eval_frequency,eval_iter,start_context,tokenizer
                       ):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1
    for epoch in range(epochs):
        model.train()
        for input_batch,target_batch in train_loader:
            input_batch,target_batch=input_batch.to(device),target_batch.to(device)
            
            optimizer.zero_grad()
            loss=calc_loss_batch(input_batch,target_batch,model,device)
            loss.backward()
            optimizer.step()
            tokens_seen +=input_batch.numel()
            global_step +=1
            
            if global_step % eval_frequency==0:
                train_loss, val_loss= evaluate_model(model,train_loader
                                                      ,val_loader,device,eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Epoch {epoch+1}, Train Loss: {train_loss:.3f} Val Loss {val_loss:.3f}")
                
        generate_and_print_sample(
            model, tokenizer, device, start_context
        )

    return train_losses, val_losses, track_tokens_seen              
            
            

In [ ]:
import time
start_time=time.time()
torch.manual_seed(123)

epoch=20
optimizer=torch.optim.Adam(model.parameters(),lr=0.001,weight_decay=0.1)
train_losses, val_losses, track_tokens_seen = train_simple_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    epochs=epoch,
    eval_frequency=5, #A
    eval_iter=5, #B
    start_context="it was no great",
    tokenizer=tokenizer
)
end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

def plot_losses(train_loss,val_loss,token_seen,epochs):
    fig,ax1=plt.subplots(figsize=(8,5))
    
    ax1.plot(epochs,train_loss,label="Training Loss")
    ax1.plot(epochs,val_loss,linestyle="-.",label="Validation Loss")
    ax1.set_xlabel("Epochs seen")
    ax1.set_ylabel("Loss")
    ax1.legend(loc="upper right")
    ax1.xaxis.set_major_locator(MaxNLocator(integer=True))
    
    ax2=ax1.twiny()
    ax2.plot(token_seen,train_loss,alpha=0)
    ax2.set_xlabel("Token Seen")
    
    fig.tight_layout()    
    plt.show()
    
plot_losses(train_losses, val_losses, track_tokens_seen,epochs=[i for i in range(0,len(train_losses))])

In [ ]:
model.to("cpu")
model.eval()

In [ ]:
tokenizer=tiktoken.get_encoding("gpt2")
context_sen="hello I am"

token_ids=generate_simpel_text(model, text_to_token_ids(context_sen, tokenizer), GPT_CONFIG_124M["context_length"], 25)

decoded_text=token_ids_to_text(token_ids,tokenizer)
print(decoded_text)

In [ ]:
def generate(model,idx,max_new_tokens,context_size,temperature=1,top_k=None,eos_id=None):
    for _ in range(max_new_tokens):
        idx_cond=idx[:, -context_size:]
        with torch.no_grad():
            logits=model(idx_cond)
        logits=logits[:,-1,:]
        
        if top_k is not None:
            top_logits,_=torch.topk(logits,20)
            min_val=top_logits[:,-1]
            logits=torch.where(logits<min_val, torch.tensor(float('-inf')).to(logits.device),logits)
        
        if temperature>0:
            logits =logits/temperature
            prob=torch.softmax(logits,dim=-1)
            idx_next=torch.multinomial(prob,num_samples=1)
        else:
            prob=torch.softmax(logits,dim=-1)
            idx_next=torch.argmax(prob,dim=-1,keepdim=True)
        
        if idx_next==eos_id:
            break
        idx=torch.cat((idx,idx_next),dim=1)
        
    return idx

            
        
            

In [ ]:
torch.manual_seed(123)

token_ids = generate(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer),
    max_new_tokens=15,
    context_size=GPT_CONFIG_124M["context_length"],
    top_k=25,
    temperature=1.4
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

In [ ]:
model=GPTModule(GPT_CONFIG_124M)
torch.save(model.state_dict(),"model.pth")

In [ ]:
model =GPTModule(GPT_CONFIG_124M)
model.load_state_dict(torch.load("model.pth"))
model.eval()


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    }, 
    "model_and_optimizer.pth"
)

In [ ]:
check_point=torch.load("model_and_optimizer.pth")

model=GPTModule(GPT_CONFIG_124M)
model.load_state_dict(check_point["model_state_dict"])

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)
optimizer.load_state_dict(check_point["optimizer_state_dict"])  
model.train(); 